In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip
from qutip.measurement import measurement_statistics_observable

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track

# Linalg libraries
import numpy as np
import h5py as hf

# Helper libraries
from dataclasses import dataclass

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
background = np.genfromtxt("data/background.csv", delimiter=",", skip_header=1)[:, 1:][:50]
signal = np.genfromtxt("data/signal.csv", delimiter=",", skip_header=1)[:, 1:][:50]

In [ ]:
background = (background - background.mean()) / background.std()
signal = (signal - signal.mean()) / signal.std()

In [ ]:
class BFieldGenerator:
    def __init__(
        self,  raw_data: np.ndarray, times: np.ndarray
        ):
        """
        Create the BField generator.

        Parameters
        ----------
        amplitude : float
            Amplitude of the magnetic field.
        frequency : float
            Frequency of the cosine wave.
        resoltion : float
            Distance between measured points.
        length : int
            Amount of time to run it for.
        """
        self.raw_data = raw_data

        self.counter = 0

        self.measured_field = [self.raw_data[0]]
        self.measured_times = []
        self.times = times

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time.
        """
        if t == self.times[self.counter + 1]:
            driving_field = self.raw_data[self.counter]

            self.measured_field.append(driving_field)      
            self.measured_times.append(t)
            self.counter += 1
        else:
            driving_field = self.measured_field[-1]

        return np.array(driving_field)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length ** 2
    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]
    length = np.sqrt(N).astype(int)
    
    # Hamiltonian - Energy splitting terms
    field = driving_field(t)
    H = 0

    # H -= field[0] * sz_list[0]
    # H -= field[0] * sz_list[1]
    # H -= field[0] * sz_list[2]
    # H -= field[0] * sx_list[0]
    # H -= field[0] * sx_list[1]
    # H -= field[0] * sx_list[2]
    H -= field[0] * sz_list[0]
    # H -= field[0] * sy_list[1]
    # H -= field[0] * sy_list[2]

    H -= field[1] * sz_list[6]
    # H -= field[1] * sz_list[7]
    # H -= field[1] * sz_list[8]
    # H -= field[1] * sx_list[6]
    # H -= field[1] * sx_list[7]
    # H -= field[1] * sx_list[8]
    # H -= field[1] * sy_list[6]
    # H -= field[1] * sy_list[7]
    # H -= field[1] * sy_list[8]

    lattice_sites = []
    for x in range(length):
        for y in range(length):
            lattice_sites.append([x, y])

    lattice_sites = np.array(lattice_sites)
    neighbours = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]]) # right, left, top, bottom
    box_l = np.array([length, length])

    # Interaction terms
    spin_coupling_term = 0
    for n, site in enumerate(lattice_sites):

        neighbours = site[None, :] + neighbours
        neighbours_folded = neighbours - np.floor(neighbours / box_l[None, :]) * box_l[None, :]
        neighbour_indices = neighbours_folded[:, 0] + length * neighbours_folded[:, 1]

        for neighbour in neighbour_indices:
            spin_coupling_term += -Jx[n] * sx_list[n] * sx_list[int(neighbour)]
            spin_coupling_term += -Jy[n] * sy_list[n] * sy_list[int(neighbour)]
            spin_coupling_term += -Jz[n] * sz_list[n] * sz_list[int(neighbour)]

    return H + spin_coupling_term * 0.5
    

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=15.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)


times = np.linspace(0, 5, signal.shape[0])

signal_field = BFieldGenerator(signal, times)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": signal_field
}

In [ ]:
signal_results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

In [ ]:
# qutip.qsave(signal_results, "signal_results")

In [ ]:
measured_signal_field = np.array(signal_field.measured_field)

plt.plot(measured_signal_field[:, 0], label="Measured")
plt.show()

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=15.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)


times = np.linspace(0, 5, background.shape[0])

bg_field = BFieldGenerator(background, times)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": bg_field
}

In [ ]:
bg_results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

In [ ]:
# qutip.qsave(bg_results, "bg_results")

In [ ]:
measured_bg_field = np.array(bg_field.measured_field)

plt.plot(background[:, 0], label="Measured")
# plt.plot(background[:, 1], label="Measured")
plt.plot(measured_bg_field[:, 0], label="Measured")

plt.show()

## Measure State Descriptions

In [ ]:
# signal_results = qutip.qload("signal_results")
# bg_results = qutip.qload("bg_results")

In [ ]:
state_dimension = 5
measure_states = 1
measurements = []

for _ in range(state_dimension):
    non_gue_matrix = qutip.rand_herm(
        2**measure_states, 
        1.0, 
        dims=[[2] * measure_states, [2] * measure_states]
    )
    measurements.append(non_gue_matrix)

In [ ]:
measure_sites = np.random.randint(0, 9, state_dimension)

In [ ]:
# Compute observables
signal_observations = np.zeros((signal.shape[0], state_dimension))
states = signal_results.states[1:]

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        measure_state = state.ptrace([measure_sites[o]])
        signal_observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

In [ ]:
# Compute observables
bg_observations = np.zeros((background.shape[0], state_dimension))
states = bg_results.states[1:]

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        measure_state = state.ptrace([measure_sites[o]])
        bg_observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)